In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures

import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from src import *

# Feature Extraction
In this notebook, we're going to create features derived from the original 7.
- Combination of polynomials up to the third order
- $\dfrac{ng}{np}$ ratio
- Normalization of density altitude
- Normalization of air density

In [2]:
# 1s
train_x, train_y = read_dataset(stage='original')
test_x = read_dataset(stage='original', testset=True)

In [3]:
# 0s
# Constants for ISA model
T0 = 288.15  # Sea-level standard temperature (K)
p0 = 101325  # Sea-level standard atmospheric pressure (Pa)
a = 0.0065  # Temperature lapse rate (K/m)
g = 9.80665  # Gravitational acceleration (m/s^2)
R = 8.3144598  # Universal gas constant (J/(mol·K))
RS = 287.05  # Specific gas constant for air (J/(kg·K))

In [4]:
# 1s
def extract_features(df)->pd.DataFrame:
    new_df = df.copy()
    poly = PolynomialFeatures(degree=3, include_bias=False)
    to_poly = df[['oat', 'mgt', 'pa', 'np', 'ng']]
    poly_features = poly.fit_transform(to_poly)
    poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(to_poly.columns))
    new_df = pd.concat([new_df, poly_df], axis=1)

    # Add NP/NG ratio feature
    new_df['np_ng_ratio'] = df['np'] / df['ng']

    # Air density formula: da = 1.2376 * pa + 118.8 * oat - 1782
    new_df['da'] = (1.2376 * df['pa']) + (118.8 * (df['oat'] + 273.15)) - 1782

    # Compute pressure at altitude (h in meters, assuming pressure altitude in dataset is in feet)
    new_df['h_m'] = df['pa'] * 0.3048  # Convert pressure altitude from feet to meters
    new_df['P'] = p0 * (1 - (a * new_df['h_m']) / T0) ** (g / (R * a))

    # Compute air density (rho), ensuring valid values
    new_df['rho'] = new_df['P'] / (RS * (df['oat'] + 273.15))

    # Normalize original features based on air density (rho)
    for col in df.columns:
        new_df[f'{col}_norm_da'] = df[col] / new_df['da']
        new_df[f'{col}_air_density'] = df[col] / new_df['rho']
    return new_df

new_train_x = extract_features(train_x)
new_test_x = extract_features(test_x)
new_test_x.head()

,trq_measured,oat,mgt,pa,ias,np,ng,oat,mgt,pa,...,mgt_norm_da,mgt_air_density,pa_norm_da,pa_air_density,ias_norm_da,ias_air_density,np_norm_da,np_air_density,ng_norm_da,ng_air_density
0,56.5,19.00,553.9,276.7584,73.6875,99.81,91.07,19.00,553.9,276.7584,...,0.016650,647.706893,0.008319,323.629398,0.002215,86.167001,0.003000,116.713531,0.002737,106.493350
1,86.2,6.75,657.9,657.4536,122.8750,100.03,97.61,6.75,657.9,657.4536,...,0.020379,1186.978551,0.020365,1186.173160,0.003806,221.690211,0.003098,180.473422,0.003023,176.107275
2,54.0,21.75,559.6,263.3472,18.1250,99.57,90.62,21.75,559.6,263.3472,...,0.016666,649.551459,0.007843,305.678267,0.000540,21.038456,0.002965,115.575123,0.002699,105.186478
3,55.4,20.75,566.8,751.0272,84.7500,99.92,91.16,20.75,566.8,751.0272,...,0.016640,1207.418120,0.022048,1599.865649,0.002488,180.537554,0.002933,212.853244,0.002676,194.192371
4,51.3,19.50,554.2,755.9040,68.5625,99.99,90.09,19.50,554.2,755.9040,...,0.016338,1182.767804,0.022285,1613.242356,0.002021,146.325365,0.002948,213.397605,0.002656,192.269129


## Export to `.csv` files

In [6]:
# 1m 0s
os.makedirs('../dataset/1-preprocessed', exist_ok=True)
new_train_x.to_csv('../dataset/1-preprocessed/X.csv', index=False)
new_test_x.to_csv('../dataset/1-preprocessed/X_test.csv', index=False)
train_y.to_csv('../dataset/1-preprocessed/y.csv', index=False)